# Versión del script para entrenar el modelo sin los rezagos de dengue

In [8]:
# ================================================================
# LIGHTGBM PARA PREDICCIÓN DE CASOS DE DENGUE
# VERSIÓN "SOLO METEOROLOGÍA" — SIN ATRIBUTOS AUTORREGRESIVOS
#
# Diferencia respecto a la versión anterior:
# 1. El algoritmo NO recibe ningún atributo derivado de "casos_dengue"
#    (nada de rezagos, medias móviles, tendencia, diferencias,
#    cambios porcentuales ni interacciones meteo × casos_lag1).
# 2. Los ÚNICOS predictores permitidos son las columnas meteorológicas
#    originales del Excel (prefijos: prec, temp, tmax, tmin, hr,
#    soi, oni, mei).
# 3. "casos_lag_1" se calcula, pero EXCLUSIVAMENTE para construir el
#    baseline naive de comparación; nunca se usa como predictor.
# 4. Se mantiene toda la lógica de robustez de la versión anterior:
#    separación temporal estricta 2021-2025 TRAIN / 2026 TEST,
#    selección de variables dentro de cada fold, umbrales de picos
#    calculados dentro de cada fold, ponderación suave de picos,
#    complejidad de LightGBM restringida, validación walk-forward,
#    número de árboles final = mediana de folds, comparación con
#    baseline naive.
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ================================================================
# 1. CONFIGURACIÓN
# ================================================================

input_file = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos"
    r"\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

output_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\3_resultados"
)

processed_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos\2_procesados"
)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

RANDOM_STATE = 42
TEST_YEAR = 2026

# Con pocas observaciones de TRAIN, limitamos el número de variables.
MAX_SELECTED_FEATURES = 12

# Menor número de ensayos para evitar optimizar excesivamente sobre pocos folds.
N_TRIALS = 40

# Máximo de árboles.
MAX_BOOST_ROUND = 1200

# Prefijos que definen qué columnas son "meteorológicas".
METEO_PREFIXES = (
    "prec", "temp", "tmax", "tmin", "hr", "soi", "oni", "mei"
)

# ================================================================
# 2. CARGA Y LIMPIEZA
# ================================================================

print("=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

df = pd.read_excel(input_file)

if "fecha" not in df.columns:
    raise ValueError("No existe la columna 'fecha'.")

if "casos_dengue" not in df.columns:
    raise ValueError("No existe la columna 'casos_dengue'.")

df["fecha"] = pd.to_datetime(df["fecha"])

# Orden cronológico.
df = df.sort_values("fecha").reset_index(drop=True)

# ------------------------------------------------
# Columnas duplicadas del Excel original.
# ------------------------------------------------

duplicated_columns = df.columns[df.columns.duplicated()].tolist()

if duplicated_columns:
    print("\nColumnas duplicadas eliminadas:")
    print(sorted(set(duplicated_columns)))
    df = df.loc[:, ~df.columns.duplicated()].copy()

target_col = "casos_dengue"
exclude_cols = ["fecha", "año", "semana_epi"]

if "año" not in df.columns:
    df["año"] = df["fecha"].dt.year

if "semana_epi" not in df.columns:
    df["semana_epi"] = (
        df["fecha"].dt.isocalendar().week.astype(int)
    )

all_candidate_cols = [
    col for col in df.columns
    if col not in exclude_cols + [target_col]
]

# ------------------------------------------------
# CORRECCIÓN CLAVE DE ESTA VERSIÓN:
# Solo se conservan como predictores las columnas meteorológicas.
# Todo lo demás (si existiera algo distinto a meteo en el Excel)
# queda fuera del conjunto de atributos.
# ------------------------------------------------

predictor_cols = [
    col for col in all_candidate_cols
    if col.startswith(METEO_PREFIXES)
]

descartadas = [
    col for col in all_candidate_cols
    if col not in predictor_cols
]

print(f"Registros originales: {len(df)}")
print(f"Predictores meteorológicos disponibles: {len(predictor_cols)}")

if descartadas:
    print(
        f"\nColumnas NO meteorológicas excluidas del modelado "
        f"({len(descartadas)}):"
    )
    print(sorted(descartadas))

print(f"Periodo: {df['fecha'].min()} → {df['fecha'].max()}")

if len(predictor_cols) == 0:
    raise ValueError(
        "No se encontró ninguna columna meteorológica con los "
        "prefijos esperados (prec, temp, tmax, tmin, hr, soi, oni, mei)."
    )

# ================================================================
# 3. PREPARACIÓN DEL DATASET (SIN INGENIERÍA AUTORREGRESIVA)
# ================================================================

print("\n" + "=" * 80)
print("PREPARACIÓN DEL DATASET — SOLO ATRIBUTOS METEOROLÓGICOS")
print("=" * 80)

df_engineered = df.copy()
y = df_engineered[target_col]

# ------------------------------------------------
# 3.1 casos_lag_1 se calcula ÚNICAMENTE para el baseline naive.
# NO se agrega a predictor_cols, por lo tanto el modelo jamás
# lo ve como atributo de entrada.
# ------------------------------------------------

df_engineered["casos_lag_1__solo_baseline"] = y.shift(1)

# new_predictor_cols es, por diseño, exactamente igual a
# predictor_cols (las columnas meteorológicas originales).
new_predictor_cols = predictor_cols.copy()

# ================================================================
# 4. LIMPIEZA Y SEPARACIÓN TEMPORAL
# ================================================================

df_engineered = df_engineered.replace(
    [np.inf, -np.inf],
    np.nan
)

# Solo se pierden filas por falta de meteo, target o el lag del baseline
# (típicamente una sola fila al inicio de la serie).
columnas_para_dropna = (
    new_predictor_cols
    + [target_col, "casos_lag_1__solo_baseline"]
)

df_engineered = df_engineered.dropna(
    subset=columnas_para_dropna
).reset_index(drop=True)

train_mask = (
    (df_engineered["año"] >= 2021)
    & (df_engineered["año"] <= 2025)
)

test_mask = df_engineered["año"] == TEST_YEAR

if test_mask.sum() == 0:
    raise ValueError("No se encontraron registros para 2026.")

print("\n" + "=" * 80)
print("SEPARACIÓN TEMPORAL")
print("=" * 80)
print(f"TRAIN: {train_mask.sum()} registros")
print(f"TEST : {test_mask.sum()} registros")
print(f"Predictores utilizados (solo meteo): {len(new_predictor_cols)}")

# ================================================================
# 5. MATRICES
# ================================================================

X_all = df_engineered[new_predictor_cols].copy()
y_all = df_engineered[target_col].copy()

X_train_full = X_all.loc[train_mask].copy()
y_train_full = y_all.loc[train_mask].copy()

X_test = X_all.loc[test_mask].copy()
y_test = y_all.loc[test_mask].copy()

fechas_test = df_engineered.loc[test_mask, "fecha"].values
años_test = df_engineered.loc[test_mask, "año"].values
semanas_test = df_engineered.loc[test_mask, "semana_epi"].values

# Solo para el baseline naive — nunca entra como feature del modelo.
casos_lag_1_all = df_engineered["casos_lag_1__solo_baseline"].copy()

# ================================================================
# 6. FUNCIONES AUXILIARES
# ================================================================

def create_walk_forward_folds(data):
    """
    Genera folds temporales.

    Ejemplo:
        Train 2022 -> Validación 2023
        Train 2022-2023 -> Validación 2024
        Train 2022-2024 -> Validación 2025

    2026 queda fuera.
    """

    years = sorted(
        data.loc[
            (data["año"] >= 2021)
            & (data["año"] <= 2025),
            "año"
        ].unique()
    )

    folds = []

    for val_year in years[1:]:
        train_years = [
            year for year in years
            if year < val_year
        ]

        train_idx = data.index[
            data["año"].isin(train_years)
        ].to_numpy()

        val_idx = data.index[
            data["año"] == val_year
        ].to_numpy()

        # Evitamos folds con entrenamiento demasiado pequeño.
        if len(train_idx) >= 40 and len(val_idx) > 0:
            folds.append(
                (
                    train_idx,
                    val_idx,
                    train_years,
                    val_year
                )
            )

    return folds


def select_features_train_only(X, y, max_features=12):
    """
    Selección de variables usando SOLO el conjunto recibido.
    Nunca debe recibir validación o test.
    Como X ya contiene únicamente columnas meteorológicas,
    la selección resultante también es exclusivamente meteorológica.
    """

    nunique = X.nunique(dropna=False)

    valid_cols = nunique[
        nunique > 1
    ].index.tolist()

    X_clean = X[valid_cols].copy()

    if len(X_clean.columns) == 0:
        raise ValueError("No quedan variables después de eliminar constantes.")

    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf.fit(X_clean, y)

    importance = pd.DataFrame({
        "Feature": X_clean.columns,
        "Importance": rf.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    n = min(
        max_features,
        len(importance)
    )

    selected = importance.head(n)["Feature"].tolist()

    return selected, importance


def create_soft_weights(y, peak_threshold, extreme_threshold):
    """
    Ponderación suave para no obligar a LightGBM a memorizar picos.

    Peso máximo limitado a 1.6.
    """

    y = np.asarray(y, dtype=float)

    weights = np.ones(len(y), dtype=float)

    weights[y >= peak_threshold] = 1.35
    weights[y >= extreme_threshold] = 1.60

    return weights


def get_fold_thresholds(y_train):
    """
    Los umbrales de pico se calculan SOLO sobre el train del fold.
    """

    y_train = np.asarray(y_train, dtype=float)

    peak = np.percentile(y_train, 80)
    extreme = np.percentile(y_train, 95)

    return peak, extreme


folds = create_walk_forward_folds(df_engineered)

print("\n" + "=" * 80)
print("FOLDS WALK-FORWARD")
print("=" * 80)

for i, (_, _, train_years, val_year) in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"Train {train_years} → "
        f"Validación {val_year}"
    )

if len(folds) < 2:
    raise ValueError(
        "No hay suficientes folds temporales para una validación robusta."
    )

# ================================================================
# 7. OPTUNA + SELECCIÓN NESTED DE VARIABLES (SOLO METEO)
# ================================================================

print("\n" + "=" * 80)
print("OPTIMIZACIÓN LIGHTGBM — SOLO ATRIBUTOS METEOROLÓGICOS")
print("=" * 80)


def objective(trial):

    params = {
        "objective": "regression_l1",
        "metric": "l1",
        "boosting_type": "gbdt",

        "num_leaves": trial.suggest_int(
            "num_leaves",
            5,
            12
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            5
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            15,
            35
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            0.05,
            0.50
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.5,
            5.0
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1.0,
            8.0
        ),

        "feature_fraction": trial.suggest_float(
            "feature_fraction",
            0.60,
            0.85
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction",
            0.60,
            0.85
        ),

        "bagging_freq": trial.suggest_int(
            "bagging_freq",
            1,
            5
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.04,
            log=True
        ),

        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }

    fold_scores = []

    for fold_number, (
        train_idx,
        val_idx,
        train_years,
        val_year
    ) in enumerate(folds, 1):

        X_tr_all = X_all.loc[train_idx]
        y_tr = y_all.loc[train_idx]

        X_val_all = X_all.loc[val_idx]
        y_val = y_all.loc[val_idx]

        # Selección de features SOLO sobre TRAIN del fold
        # (conjunto ya restringido a variables meteorológicas).
        selected_fold_features, _ = select_features_train_only(
            X_tr_all,
            y_tr,
            MAX_SELECTED_FEATURES
        )

        X_tr = X_tr_all[selected_fold_features]
        X_val = X_val_all[selected_fold_features]

        peak_threshold_fold, extreme_threshold_fold = (
            get_fold_thresholds(y_tr)
        )

        weights_tr = create_soft_weights(
            y_tr.values,
            peak_threshold_fold,
            extreme_threshold_fold
        )

        dtrain = lgb.Dataset(
            X_tr,
            label=y_tr,
            weight=weights_tr
        )

        dval = lgb.Dataset(
            X_val,
            label=y_val,
            reference=dtrain
        )

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=MAX_BOOST_ROUND,
            valid_sets=[dval],
            valid_names=["validation"],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=60,
                    verbose=False
                ),
                lgb.log_evaluation(0)
            ]
        )

        pred = model.predict(
            X_val,
            num_iteration=model.best_iteration
        )

        general_mae = mean_absolute_error(
            y_val,
            pred
        )

        peak_mask = (
            y_val.values >= peak_threshold_fold
        )

        if peak_mask.sum() > 0:
            peak_mae = mean_absolute_error(
                y_val.values[peak_mask],
                pred[peak_mask]
            )
        else:
            peak_mae = general_mae

        if peak_mask.sum() > 0:
            underestimation = np.mean(
                np.maximum(
                    0,
                    y_val.values[peak_mask]
                    - pred[peak_mask]
                )
            )
        else:
            underestimation = 0.0

        combined_score = (
            0.80 * general_mae
            + 0.15 * peak_mae
            + 0.05 * underestimation
        )

        fold_scores.append(combined_score)

    return float(np.mean(fold_scores))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\nMejores parámetros:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

print(
    f"\nMejor score walk-forward: "
    f"{study.best_value:.4f}"
)

# ================================================================
# 8. SELECCIÓN FINAL DE FEATURES (SOLO METEO)
# ================================================================

print("\n" + "=" * 80)
print("SELECCIÓN FINAL DE FEATURES — SOLO TRAIN 2021-2025 / SOLO METEO")
print("=" * 80)

selected_features_rf, feature_importance = (
    select_features_train_only(
        X_train_full,
        y_train_full,
        MAX_SELECTED_FEATURES
    )
)

print(
    f"Predictores meteorológicos disponibles: "
    f"{len(new_predictor_cols)}"
)

print(
    f"Features seleccionadas: "
    f"{len(selected_features_rf)}"
)

print("\nFeatures seleccionadas:")

for i, feature in enumerate(
    selected_features_rf,
    1
):
    row = feature_importance[
        feature_importance["Feature"] == feature
    ].iloc[0]

    print(
        f"{i:2d}. "
        f"{feature:<40} "
        f"{row['Importance']:.6f}"
    )

features_file = os.path.join(
    output_dir,
    "atributos_seleccionados_rf_solo_meteo.xlsx"
)

feature_importance.to_excel(
    features_file,
    index=False
)

X_train_full = X_train_full[
    selected_features_rf
].copy()

X_test = X_test[
    selected_features_rf
].copy()

# ================================================================
# 9. ESTIMACIÓN DEL NÚMERO DE ÁRBOLES
# ================================================================

print("\n" + "=" * 80)
print("ESTIMACIÓN DE ITERACIONES FINALES")
print("=" * 80)

best_iterations = []

for fold_number, (
    train_idx,
    val_idx,
    train_years,
    val_year
) in enumerate(folds, 1):

    X_tr_all = X_all.loc[train_idx]
    y_tr = y_all.loc[train_idx]

    X_val_all = X_all.loc[val_idx]
    y_val = y_all.loc[val_idx]

    selected_fold_features, _ = select_features_train_only(
        X_tr_all,
        y_tr,
        MAX_SELECTED_FEATURES
    )

    X_tr = X_tr_all[selected_fold_features]
    X_val = X_val_all[selected_fold_features]

    peak_threshold_fold, extreme_threshold_fold = (
        get_fold_thresholds(y_tr)
    )

    weights_tr = create_soft_weights(
        y_tr.values,
        peak_threshold_fold,
        extreme_threshold_fold
    )

    dtrain = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=weights_tr
    )

    dval = lgb.Dataset(
        X_val,
        label=y_val,
        reference=dtrain
    )

    fold_params = {
        "objective": "regression_l1",
        "metric": "l1",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        **best_params
    }

    fold_model = lgb.train(
        fold_params,
        dtrain,
        num_boost_round=MAX_BOOST_ROUND,
        valid_sets=[dval],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=60,
                verbose=False
            ),
            lgb.log_evaluation(0)
        ]
    )

    best_iterations.append(
        fold_model.best_iteration
    )

print(
    f"Mejores iteraciones por fold: "
    f"{best_iterations}"
)

final_num_boost_round = int(
    np.median(best_iterations)
)

final_num_boost_round = max(
    30,
    min(
        final_num_boost_round,
        600
    )
)

print(
    f"Número de iteraciones finales: "
    f"{final_num_boost_round}"
)

# ================================================================
# 10. ENTRENAMIENTO FINAL
# ================================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINAL")
print("=" * 80)

peak_threshold_final, extreme_threshold_final = (
    get_fold_thresholds(y_train_full)
)

print(
    f"Umbral pico TRAIN: "
    f"{peak_threshold_final:.3f}"
)

print(
    f"Umbral extremo TRAIN: "
    f"{extreme_threshold_final:.3f}"
)

train_weights = create_soft_weights(
    y_train_full.values,
    peak_threshold_final,
    extreme_threshold_final
)

final_params = {
    "objective": "regression_l1",
    "metric": "l1",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    **best_params
}

dtrain_final = lgb.Dataset(
    X_train_full,
    label=y_train_full,
    weight=train_weights
)

model = lgb.train(
    final_params,
    dtrain_final,
    num_boost_round=final_num_boost_round,
    callbacks=[
        lgb.log_evaluation(0)
    ]
)

# ================================================================
# 11. PREDICCIONES
# ================================================================

y_train_pred = model.predict(
    X_train_full
)

y_test_pred = model.predict(
    X_test
)

y_train_pred = np.clip(
    y_train_pred,
    0,
    None
)

y_test_pred = np.clip(
    y_test_pred,
    0,
    None
)

# ================================================================
# 12. BASELINE NAIVE
# (usa casos_lag_1, calculado aparte — NUNCA fue predictor del modelo)
# ================================================================

naive_test_pred = casos_lag_1_all.loc[test_mask].values

naive_test_pred = np.clip(
    naive_test_pred,
    0,
    None
)

naive_mae = mean_absolute_error(
    y_test,
    naive_test_pred
)

naive_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        naive_test_pred
    )
)

naive_r2 = r2_score(
    y_test,
    naive_test_pred
)

# ================================================================
# 13. MÉTRICAS
# ================================================================

train_mae = mean_absolute_error(
    y_train_full,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train_full,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_r2 = r2_score(
    y_train_full,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

peak_mask_train = (
    y_train_full.values
    >= peak_threshold_final
)

peak_mask_test = (
    y_test.values
    >= peak_threshold_final
)

peak_mae_train = (
    mean_absolute_error(
        y_train_full.values[peak_mask_train],
        y_train_pred[peak_mask_train]
    )
    if peak_mask_train.sum() > 0
    else np.nan
)

peak_mae_test = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        y_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

bias_train = np.mean(
    y_train_pred
    - y_train_full.values
)

bias_test = np.mean(
    y_test_pred
    - y_test.values
)

naive_peak_mae = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        naive_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

if naive_mae > 0:
    improvement_vs_naive = (
        (naive_mae - test_mae)
        / naive_mae
    ) * 100
else:
    improvement_vs_naive = np.nan

generalization_gap_mae = (
    test_mae - train_mae
)

generalization_ratio_mae = (
    test_mae / max(train_mae, 1e-9)
)

# ================================================================
# 14. RESULTADOS
# ================================================================

print("\n" + "=" * 80)
print("RESULTADOS FINALES — VERSIÓN SOLO METEOROLOGÍA")
print("=" * 80)

print(f"MAE Train       : {train_mae:.4f}")
print(f"MAE Test        : {test_mae:.4f}")
print(f"Brecha MAE      : {generalization_gap_mae:.4f}")
print(f"Ratio Test/Train: {generalization_ratio_mae:.2f}x")

print(f"\nPeak MAE Train  : {peak_mae_train:.4f}")
print(f"Peak MAE Test   : {peak_mae_test:.4f}")

print(f"\nRMSE Train      : {train_rmse:.4f}")
print(f"RMSE Test       : {test_rmse:.4f}")

print(f"\nR² Train        : {train_r2:.4f}")
print(f"R² Test         : {test_r2:.4f}")

print(f"\nBias Train      : {bias_train:.4f}")
print(f"Bias Test       : {bias_test:.4f}")

print("\nBASELINE NAIVE — y(t) = y(t-1)")
print(f"MAE Test        : {naive_mae:.4f}")
print(f"RMSE Test       : {naive_rmse:.4f}")
print(f"R² Test         : {naive_r2:.4f}")
print(f"Peak MAE Test   : {naive_peak_mae:.4f}")

print(
    f"\nMejora frente a baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("=" * 80)

# ================================================================
# 15. IMPORTANCIA LIGHTGBM
# ================================================================

lgb_importance = pd.DataFrame({
    "Feature": selected_features_rf,
    "Importance_gain": model.feature_importance(
        importance_type="gain"
    ),
    "Importance_split": model.feature_importance(
        importance_type="split"
    )
}).sort_values(
    "Importance_gain",
    ascending=False
)

# ================================================================
# 16. TABLA DE PREDICCIONES
# ================================================================

pred_df = pd.DataFrame({
    "fecha": fechas_test,
    "año": años_test,
    "semana_epi": semanas_test,
    "casos_reales": y_test.values,
    "predicciones_lightgbm": y_test_pred,
    "prediccion_naive": naive_test_pred
})

pred_df["error"] = (
    pred_df["predicciones_lightgbm"]
    - pred_df["casos_reales"]
)

pred_df["error_absoluto"] = (
    np.abs(pred_df["error"])
)

pred_df["es_pico"] = (
    pred_df["casos_reales"]
    >= peak_threshold_final
)

# ================================================================
# 17. ANÁLISIS POR RANGO
# ================================================================

max_case = max(
    200,
    int(
        np.ceil(
            y_test.max() / 50
        ) * 50
    )
)

bins = [
    -np.inf,
    5,
    10,
    20,
    50,
    100,
    200,
    max_case
]

bins_unique = sorted(
    set(bins)
)

labels_unique = [
    f"{bins_unique[i]}-{bins_unique[i+1]}"
    for i in range(
        len(bins_unique) - 1
    )
]

pred_df["rango_casos"] = pd.cut(
    pred_df["casos_reales"],
    bins=bins_unique,
    labels=labels_unique,
    include_lowest=True
)

error_analysis_rows = []

for label in pred_df[
    "rango_casos"
].dropna().unique():

    mask = (
        pred_df["rango_casos"] == label
    )

    if mask.sum() > 0:

        error_analysis_rows.append({
            "Rango": str(label),
            "Count": int(mask.sum()),
            "MAE": pred_df.loc[
                mask,
                "error_absoluto"
            ].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    pred_df.loc[
                        mask,
                        "error"
                    ] ** 2
                )
            ),
            "Bias": pred_df.loc[
                mask,
                "error"
            ].mean(),
            "Max_Error": pred_df.loc[
                mask,
                "error_absoluto"
            ].max()
        })

error_analysis = pd.DataFrame(
    error_analysis_rows
)

# ================================================================
# 18. GUARDAR DATASET PROCESADO
# ================================================================

processed_file = os.path.join(
    processed_dir,
    "dataset_procesado_solo_meteo.xlsx"
)

columns_to_save = (
    exclude_cols
    + [target_col]
    + selected_features_rf
)

columns_to_save = [
    col for col in columns_to_save
    if col in df_engineered.columns
]

df_final = df_engineered[
    columns_to_save
].copy()

df_final.to_excel(
    processed_file,
    index=False
)

# ================================================================
# 19. GUARDAR RESULTADOS EXCEL
# ================================================================

excel_file = os.path.join(
    output_dir,
    "resultados_modelo_solo_meteo.xlsx"
)

metrics_df = pd.DataFrame({
    "Métrica": [
        "MAE",
        "RMSE",
        "R²",
        "Peak MAE 80% TRAIN",
        "Bias",
        "Brecha MAE Test-Train",
        "Ratio MAE Test/Train"
    ],
    "LightGBM Train": [
        train_mae,
        train_rmse,
        train_r2,
        peak_mae_train,
        bias_train,
        generalization_gap_mae,
        generalization_ratio_mae
    ],
    "LightGBM Test 2026": [
        test_mae,
        test_rmse,
        test_r2,
        peak_mae_test,
        bias_test,
        generalization_gap_mae,
        generalization_ratio_mae
    ],
    "Baseline Naive Test 2026": [
        naive_mae,
        naive_rmse,
        naive_r2,
        naive_peak_mae,
        np.mean(
            naive_test_pred
            - y_test.values
        ),
        np.nan,
        np.nan
    ]
})

params_df = pd.DataFrame({
    "Parámetro": list(
        best_params.keys()
    ),
    "Valor": [
        str(v)
        for v in best_params.values()
    ]
})

walk_forward_rows = []

for i, fold in enumerate(
    folds,
    1
):
    walk_forward_rows.append({
        "Fold": i,
        "Train": str(fold[2]),
        "Validacion": fold[3],
        "Best_iteration": (
            best_iterations[i - 1]
            if i - 1 < len(best_iterations)
            else np.nan
        )
    })

walk_forward_df = pd.DataFrame(
    walk_forward_rows
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    metrics_df.to_excel(
        writer,
        sheet_name="Metricas",
        index=False
    )

    pred_df.to_excel(
        writer,
        sheet_name="Predicciones_Test",
        index=False
    )

    feature_importance.to_excel(
        writer,
        sheet_name="Features_RF",
        index=False
    )

    lgb_importance.to_excel(
        writer,
        sheet_name="Features_LightGBM",
        index=False
    )

    params_df.to_excel(
        writer,
        sheet_name="Parametros",
        index=False
    )

    walk_forward_df.to_excel(
        writer,
        sheet_name="Walk_Forward",
        index=False
    )

    error_analysis.to_excel(
        writer,
        sheet_name="Analisis_Errores",
        index=False
    )

# ================================================================
# 20. GUARDAR MODELO
# ================================================================

model_file = os.path.join(
    output_dir,
    "modelo_lightgbm_final_solo_meteo.txt"
)

model.save_model(
    model_file
)

# ================================================================
# 21. GRÁFICO
# ================================================================

fig, ax = plt.subplots(
    figsize=(16, 6)
)

ax.plot(
    fechas_test,
    y_test.values,
    label="Real",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    y_test_pred,
    label="LightGBM (solo meteo)",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    naive_test_pred,
    label="Baseline naive",
    linewidth=1.2,
    linestyle="--"
)

ax.scatter(
    fechas_test[peak_mask_test],
    y_test.values[peak_mask_test],
    s=30,
    label="Picos definidos con TRAIN"
)

ax.set_xlabel("Fecha")
ax.set_ylabel("Casos de dengue")

ax.set_title(
    f"Predicción 2026 (solo meteo) — "
    f"MAE LightGBM = {test_mae:.2f} | "
    f"MAE Naive = {naive_mae:.2f}"
)

ax.legend()
ax.grid(
    True,
    alpha=0.3
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plot_file = os.path.join(
    output_dir,
    "comparativa_test_2026_solo_meteo.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ================================================================
# 22. RESUMEN FINAL
# ================================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL — MODELO ENTRENADO SOLO CON METEOROLOGÍA")
print("=" * 80)

print(
    f"✓ Registros TRAIN: "
    f"{len(X_train_full)}"
)

print(
    f"✓ Registros TEST 2026: "
    f"{len(X_test)}"
)

print(
    f"✓ Predictores meteorológicos disponibles: "
    f"{len(new_predictor_cols)}"
)

print(
    f"✓ Features finales seleccionadas: "
    f"{len(selected_features_rf)}"
)

print(
    f"✓ MAE Train: "
    f"{train_mae:.2f}"
)

print(
    f"✓ MAE Test 2026: "
    f"{test_mae:.2f}"
)

print(
    f"✓ Brecha MAE: "
    f"{generalization_gap_mae:.2f}"
)

print(
    f"✓ Ratio Test/Train: "
    f"{generalization_ratio_mae:.2f}x"
)

print(
    f"✓ Peak MAE Test 2026: "
    f"{peak_mae_test:.2f}"
)

print(
    f"✓ R² Test 2026: "
    f"{test_r2:.4f}"
)

print(
    f"✓ MAE Baseline Naive: "
    f"{naive_mae:.2f}"
)

print(
    f"✓ Mejora frente a baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("\nArchivos generados:")

print(
    f"  - Dataset procesado: "
    f"{processed_file}"
)

print(
    f"  - Features RF: "
    f"{features_file}"
)

print(
    f"  - Resultados Excel: "
    f"{excel_file}"
)

print(
    f"  - Modelo LightGBM: "
    f"{model_file}"
)

print(
    f"  - Gráfico: "
    f"{plot_file}"
)

print("=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)


CARGANDO DATOS
Registros originales: 270
Predictores meteorológicos disponibles: 65

Columnas NO meteorológicas excluidas del modelado (103):
['casos_dengue_lag_1', 'casos_dengue_lag_10', 'casos_dengue_lag_11', 'casos_dengue_lag_12', 'casos_dengue_lag_2', 'casos_dengue_lag_3', 'casos_dengue_lag_4', 'casos_dengue_lag_5', 'casos_dengue_lag_6', 'casos_dengue_lag_7', 'casos_dengue_lag_8', 'casos_dengue_lag_9', 'dias_lluvia', 'dias_lluvia_lag_1', 'dias_lluvia_lag_10', 'dias_lluvia_lag_11', 'dias_lluvia_lag_12', 'dias_lluvia_lag_2', 'dias_lluvia_lag_3', 'dias_lluvia_lag_4', 'dias_lluvia_lag_5', 'dias_lluvia_lag_6', 'dias_lluvia_lag_7', 'dias_lluvia_lag_8', 'dias_lluvia_lag_9', 'hum_esp', 'hum_esp_lag_1', 'hum_esp_lag_10', 'hum_esp_lag_11', 'hum_esp_lag_12', 'hum_esp_lag_2', 'hum_esp_lag_3', 'hum_esp_lag_4', 'hum_esp_lag_5', 'hum_esp_lag_6', 'hum_esp_lag_7', 'hum_esp_lag_8', 'hum_esp_lag_9', 'hum_rel', 'hum_rel_lag_1', 'hum_rel_lag_10', 'hum_rel_lag_11', 'hum_rel_lag_12', 'hum_rel_lag_2', 'hu

Best trial: 32. Best value: 24.0876: 100%|██████████| 40/40 [00:57<00:00,  1.44s/it]



Mejores parámetros:
  num_leaves: 8
  max_depth: 4
  min_child_samples: 26
  min_split_gain: 0.15469259733922144
  reg_alpha: 3.3298735515904205
  reg_lambda: 2.5075013955620604
  feature_fraction: 0.6468576268424348
  bagging_fraction: 0.7899570376882424
  bagging_freq: 4
  learning_rate: 0.02168608618039519

Mejor score walk-forward: 24.0876

SELECCIÓN FINAL DE FEATURES — SOLO TRAIN 2021-2025 / SOLO METEO
Predictores meteorológicos disponibles: 65
Features seleccionadas: 12

Features seleccionadas:
 1. temp_max_lag_2                           0.031532
 2. temp_max                                 0.028224
 3. temp_max_lag_5                           0.027106
 4. temp_max_lag_3                           0.026601
 5. temp_max_lag_1                           0.026212
 6. temp_max_lag_4                           0.025745
 7. temp                                     0.025420
 8. soi_lag_8                                0.022011
 9. temp_max_lag_6                           0.021690
10. tem